In [0]:
import pandas as pd

In [0]:
df_circuit = spark.read.csv("/Volumes/gr5069/raw/f1_data/circuits.csv", header=True)

In [0]:
from pyspark.sql.functions import concat_ws, datediff, to_date, floor, col, avg, min, max, when, reduce, col, upper, substring, year
from pyspark.sql.window import Window

In [0]:
display(df_circuit)

In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
display(df_pitstops.head(5))

In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header=True)

In [0]:
df = df_laptimes.withColumn("milliseconds", col("milliseconds").cast("int"))
driver_avg = df.groupBy("raceId", "driverId").agg(
    floor(avg("milliseconds")).alias("avg_pit_time"),
).orderBy("raceId")

display(driver_avg.head(5))

In [0]:
display(df)

In [0]:
df.write.mode('overwrite').saveAsTable('gr5069.inclass.zl3652_predDF_final')

In [0]:
race_stats = df.groupBy("raceId").agg(
    min("milliseconds").alias("fastest_pit_stop in ms"),
    max("milliseconds").alias("slowest_pit_stop in ms"),
).orderBy("raceId")

display(race_stats.head(5))

In [0]:
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

In [0]:
drivers_pitstops = driver_avg.join(
    drivers.select("driverId", "forename", "surname"),
    on="driverId"
).orderBy("raceId")


display(drivers_pitstops.head(5))

In [0]:
driver_standing = spark.read.csv("/Volumes/gr5069/raw/f1_data/driver_standings.csv", header=True)
display(driver_standing.head(5))

In [0]:
pitstop_position = drivers_pitstops.join(
    driver_standing.select("raceId", "driverId", "position"),
    on=["raceId", "driverId"]
).withColumn("position", col("position").cast("int")).orderBy( "position", "avg_pit_time")

display(pitstop_position.head(5))

In [0]:
race_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
display(race_results.head(5))

In [0]:
race_results_renamed = race_results.withColumnRenamed("position", "finish_position")

joined = pitstop_position.join(
    race_results_renamed,
    on=["raceId", "driverId"],
    how="inner"
)

In [0]:
display(race_results_renamed)

joined = joined.drop("finish_position", "positionText", "points", "time", "milliseconds", "resultId", "number", "forename", "surname", "statusId", "positionOrder")
display(joined)

Perform a train/test split.

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
df = joined.toPandas()
X_train, X_test, y_train, y_test = train_test_split(df.drop(["position"], axis=1), df[["position"]].values.ravel(), random_state=42)

In [0]:
from pyspark.sql.functions import col, when, mean, expr, split
from pyspark.sql.types import StringType

# Step 1: Replace "N" and "\N" with null in string columns
df_cleaned = joined
for f in joined.schema.fields:
    if isinstance(f.dataType, StringType):
        df_cleaned = df_cleaned.withColumn(
            f.name,
            when(col(f.name).isin("N", "\\N"), None).otherwise(col(f.name))
        )

# Step 2a: Convert fastestLapTime ("M:SS.sss") to milliseconds
parts = split(col("fastestLapTime"), ":")
df_cleaned = df_cleaned.withColumn(
    "fastestLapTime",
    when(
        col("fastestLapTime").isNull(), None
    ).otherwise(
        floor((parts.getItem(0).cast("double") * 60 + parts.getItem(1).cast("double")) * 1000)
    )
)

# Step 2b: Safely cast the remaining numeric columns
for c in ["fastestLapSpeed", "Rank"]:
    df_cleaned = df_cleaned.withColumn(c, expr(f"try_cast(`{c}` as double)"))

# Step 3: Compute the mean for each numeric column
numeric_cols = ["fastestLapTime", "fastestLapSpeed", "Rank"]
means = df_cleaned.select([mean(col(c)).alias(c) for c in numeric_cols]).first().asDict()

# Step 4: Fill nulls with the means
df_filled = df_cleaned.fillna(means)

display(df_filled)

In [0]:
pip install mlflow

In [0]:
!pip install typing_extensions --upgrade

In [0]:
import mlflow
import mlflow.sklearn
import numpy as np
from pyspark.sql.functions import col, when, expr, split, floor
from pyspark.sql.types import StringType
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# -----------------------------------------------------------------------------
# Step 1: Replace "N" and "\N" with null in all string columns
# -----------------------------------------------------------------------------
df_cleaned = joined
for f in joined.schema.fields:
    if isinstance(f.dataType, StringType):
        df_cleaned = df_cleaned.withColumn(
            f.name,
            when(col(f.name).isin("N", "\\N"), None).otherwise(col(f.name))
        )

# -----------------------------------------------------------------------------
# Step 2a: Convert fastestLapTime ("M:SS.sss") to milliseconds, floored
# -----------------------------------------------------------------------------
parts = split(col("fastestLapTime"), ":")
df_cleaned = df_cleaned.withColumn(
    "fastestLapTime",
    when(
        col("fastestLapTime").isNull(), None
    ).otherwise(
        floor((parts.getItem(0).cast("double") * 60 + parts.getItem(1).cast("double")) * 1000)
    )
)

# -----------------------------------------------------------------------------
# Step 2b: Safely cast remaining numeric columns
# -----------------------------------------------------------------------------
numeric_cols = ["fastestLapTime", "fastestLapSpeed", "Rank"]
for c in ["fastestLapSpeed", "Rank"]:
    df_cleaned = df_cleaned.withColumn(c, expr(f"try_cast(`{c}` as double)"))

# -----------------------------------------------------------------------------
# Step 3: Define your features and target, then cast features to numeric too
# -----------------------------------------------------------------------------
target_col = "Position"  # <-- change to your actual target column
feature_cols = ["fastestLapTime", "fastestLapSpeed", "Rank"] 

# Cast target to numeric as well
df_cleaned = df_cleaned.withColumn(target_col, expr(f"try_cast(`{target_col}` as double)"))

# Cast any remaining feature columns that aren't already numeric
for c in feature_cols:
    df_cleaned = df_cleaned.withColumn(c, expr(f"try_cast(`{c}` as double)"))

# -----------------------------------------------------------------------------
# Step 4: Move to pandas for sklearn
# -----------------------------------------------------------------------------
pdf = df_cleaned.select(feature_cols + [target_col]).toPandas()

# Drop rows where the target is missing — can't train on those
pdf = pdf.dropna(subset=[target_col])

X = pdf[feature_cols]
y = pdf[target_col].astype(float)

# -----------------------------------------------------------------------------
# Step 5: Train/test split, then fill feature NaNs using TRAIN means only
# -----------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_means = X_train.mean(numeric_only=True)
X_train = X_train.fillna(train_means)
X_test = X_test.fillna(train_means)  # use train means, not test means

# -----------------------------------------------------------------------------
# Step 6: Train, log, and evaluate with MLflow
# -----------------------------------------------------------------------------
with mlflow.start_run(run_name="Basic RF Experiment") as run:
    rf = RandomForestRegressor(random_state=42)
    rf.fit(X_train, y_train)
    predictions = rf.predict(X_test)

    mlflow.sklearn.log_model(rf, "random-forest-model")

    mse = mean_squared_error(y_test, predictions)
    print(f"  mse: {mse}")
    mlflow.log_metric("mse", mse)

    # Log a few useful params for reproducibility
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("train_rows", len(X_train))

    run_id = run.info.run_id
    experiment_id = run.info.experiment_id
    print(f"Inside MLflow Run with run_id {run_id} and experiment_id {experiment_id}")

In [0]:


# -----------------------------------------------------------------------------
# Step 1: Replace "N" and "\N" with null in all string columns
# -----------------------------------------------------------------------------
df_cleaned = joined
for f in joined.schema.fields:
    if isinstance(f.dataType, StringType):
        df_cleaned = df_cleaned.withColumn(
            f.name,
            when(col(f.name).isin("N", "\\N"), None).otherwise(col(f.name))
        )

# -----------------------------------------------------------------------------
# Step 2a: Convert fastestLapTime ("M:SS.sss") to milliseconds, floored
# -----------------------------------------------------------------------------
parts = split(col("fastestLapTime"), ":")
df_cleaned = df_cleaned.withColumn(
    "fastestLapTime",
    when(
        col("fastestLapTime").isNull(), None
    ).otherwise(
        floor((parts.getItem(0).cast("double") * 60 + parts.getItem(1).cast("double")) * 1000)
    )
)

# -----------------------------------------------------------------------------
# Step 2b: Safely cast remaining numeric columns
# -----------------------------------------------------------------------------
numeric_cols = ["fastestLapTime", "fastestLapSpeed", "Rank"]
for c in ["fastestLapSpeed", "Rank"]:
    df_cleaned = df_cleaned.withColumn(c, expr(f"try_cast(`{c}` as double)"))

# -----------------------------------------------------------------------------
# Step 3: Define your features and target, then cast features to numeric too
# -----------------------------------------------------------------------------
target_col = "Position"  # <-- change to your actual target column
feature_cols = ["fastestLapTime", "fastestLapSpeed", "Rank"]

# Cast target to numeric as well
df_cleaned = df_cleaned.withColumn(target_col, expr(f"try_cast(`{target_col}` as double)"))

# Cast any remaining feature columns that aren't already numeric
for c in feature_cols:
    df_cleaned = df_cleaned.withColumn(c, expr(f"try_cast(`{c}` as double)"))

# -----------------------------------------------------------------------------
# Step 4: Move to pandas for sklearn
# -----------------------------------------------------------------------------
pdf = df_cleaned.select(feature_cols + [target_col]).toPandas()

# Drop rows where the target is missing — can't train on those
pdf = pdf.dropna(subset=[target_col])

X = pdf[feature_cols]
y = pdf[target_col].astype(float)

# -----------------------------------------------------------------------------
# Step 5: Train/test split, then fill feature NaNs with -1 sentinel
# -----------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.fillna(-1)
X_test = X_test.fillna(-1)

# -----------------------------------------------------------------------------
# Step 6: Train, log, and evaluate with MLflow
# -----------------------------------------------------------------------------
with mlflow.start_run(run_name="Basic RF Experiment") as run:
    rf = RandomForestRegressor(random_state=42)
    rf.fit(X_train, y_train)
    predictions = rf.predict(X_test)

    mlflow.sklearn.log_model(rf, "random-forest-model")

    mse = mean_squared_error(y_test, predictions)
    print(f"  mse: {mse}")
    mlflow.log_metric("mse", mse)

    # Log a few useful params for reproducibility
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("train_rows", len(X_train))

    run_id = run.info.run_id
    experiment_id = run.info.experiment_id
    print(f"Inside MLflow Run with run_id {run_id} and experiment_id {experiment_id}")

    #spark_pred_df = spark.createDataFrame(predictions)
    #spark_pred_df.write.mode("overwrite").saveAsTable("gr5069.zl3652.pred_df")

In [0]:

run_id = run.info.run_id  # from your MLflow run
artifact_uri = mlflow.get_run(run_id).info.artifact_uri
print(f"Artifacts are stored at: {artifact_uri}")

# To display a logged artifact (e.g., a plot) in Databricks:
from PIL import Image
import IPython.display as display_img

img = Image.open("actual_vs_predicted.png")  # replace with your artifact filename
display_img.display(img)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot 1: Feature importance bar plot
plt.figure(figsize=(8, 6))
importances = model.feature_importances_ if hasattr(model, 'feature_importances_') and hasattr(model.feature_importances_, '__len__') else None


plt.title("Feature Importances")
plt.ylabel("Importance")
plt.xlabel("Feature")
plt.tight_layout()
feature_importance_path = "feature_importance.png"
plt.savefig(feature_importance_path)
plt.close()
mlflow.log_artifact(feature_importance_path)
display(feature_importance_path)

# Plot 2: Prediction error scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(predictions, y_test - predictions, alpha=0.5)
plt.xlabel("Predicted Position")
plt.ylabel("Prediction Error")
plt.title("Prediction Error vs Predicted Position")
plt.grid(True)
error_scatter_path = "prediction_error_scatter.png"
plt.savefig(error_scatter_path)
plt.close()
mlflow.log_artifact(error_scatter_path)
display(error_scatter_path)



 2. Create an experiment setup where - for each run - you log:

the hyperparameters used in the model
the model itself
every possible metric from the model you chose
at least two artifacts (plots, or csv files)

In [0]:
display(pdf)

In [0]:
!pip install xgboost
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

with mlflow.start_run(run_name="XGBoost Experiment") as run:
    model = xgb.XGBRegressor(
        n_estimators=2000,
        learning_rate=0.01,
        max_depth=9,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mlflow.xgboost.log_model(model, "xgboost-model")

    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    print(f"  mse: {mse}")
    print(f"  r2:  {r2}")

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    mlflow.log_param("n_estimators", 2000)
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("max_depth", 9)

    run_id = run.info.run_id
    print(f"Run id: {run_id}")

    # Store predictions in your own database table
    pred_df = X_test.copy()
    pred_df["prediction"] = predictions
    pred_df["run_id"] = run_id

    # Plot 1: Actual vs Predicted scatter plot
    plt.figure(figsize=(8, 6))
    plt.scatter(y_test, predictions, alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted Position")
    plt.grid(True)
    scatter_path = "actual_vs_predicted.png"
    plt.savefig(scatter_path)
    plt.close()
    mlflow.log_artifact(scatter_path)

    # Plot 2: Residuals histogram
    residuals = y_test - predictions
    plt.figure(figsize=(8, 6))
    sns.histplot(residuals, bins=30, kde=True)
    plt.xlabel("Residuals")
    plt.title("Residuals Distribution")
    plt.grid(True)
    residuals_path = "residuals_hist.png"
    plt.savefig(residuals_path)
    plt.close()
    mlflow.log_artifact(residuals_path)

In [0]:
import matplotlib.pyplot as plt
import shap

rf.fit(X_train, y_train)
explainer = shap.Explainer(rf, X_train)
shap_values = explainer(X_test)

plt.figure()
shap.summary_plot(shap_values, X_test, show=True)

In [0]:
spark_pred_df = spark.createDataFrame(pred_df)
spark_pred_df.write.mode("overwrite").saveAsTable("gr5069.zl3652.pred_df")

the hyperparameters used in the model
the model itself
every possible metric from the model you chose
at least two artifacts